##### How to Get `Size / Length` of Array & Map Column?
##### How to `Filter` based on the `Size` of Array Type Column?

##### array_size()

- is used to find **how many elements** are present in an **array column**.
- **array_size(col):** `Number of elements in array`.
- **Empty array** → `0`
- **Null array** → `null`
- Works only on **ArrayType columns**.

**Syntax**

     array_size(array)

**Arguments**
- **array:** An ARRAY expression.

**Returns**
- An **INTEGER**.

**Topics Covered:**
- count elements in an `array & map` column
- `filter` condition
- `conditional logic`
- `nested arrays`
- Difference between `size() & array_size()`

##### 1) count elements in an `array & map` column

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, lit, array, array_size, size
from pyspark.sql.types import IntegerType, StringType, ArrayType, StructType, StructField

In [0]:
data = [
    (1, ["Harish", "561321", "Bangalore", "India"], {'product':'bmw', 'color':'brown', 'type':'sedan'}),
    (2, ["Kishore", "557890", ""], {'product':'audi', 'color':None, 'type':'truck'}),
    (3, ["Ravi",  "Kumar", "Azure", "678435", None, "India"], {'product':'volvo', 'color':'', 'type':''}),
    (4, ["Rajesh", ""], {'product':'toyota', 'color':'white'}),
    (5, ["Rajesh"], {'product':'mercedes'}),
    (6, ["", "", ""], {'product':'hyundai', 'color':'white', 'type':'suv', 'Age':None}),
    (7, [], {}),
    (8, None, None)
]

df = spark.createDataFrame(data, ["id", "address", "properties"])
display(df)
df.printSchema()

id,address,properties
1,"List(Harish, 561321, Bangalore, India)","Map(product -> bmw, color -> brown, type -> sedan)"
2,"List(Kishore, 557890, )","Map(product -> audi, color -> null, type -> truck)"
3,"List(Ravi, Kumar, Azure, 678435, null, India)","Map(product -> volvo, color -> , type -> )"
4,"List(Rajesh, )","Map(product -> toyota, color -> white)"
5,List(Rajesh),Map(product -> mercedes)
6,"List(, , )","Map(product -> hyundai, color -> white, type -> suv, Age -> null)"
7,List(),Map()
8,null,null


root
 |-- id: long (nullable = true)
 |-- address: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- properties: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)



In [0]:
df_size = (df.withColumn("address_count", array_size("address"))  # Array Column
             .withColumn("properties_count", size("properties"))  # Map Column
            )
display(df_size)
df_size.printSchema()

id,address,properties,address_count,properties_count
1,"List(Harish, 561321, Bangalore, India)","Map(product -> bmw, color -> brown, type -> sedan)",4,3
2,"List(Kishore, 557890, )","Map(product -> audi, color -> null, type -> truck)",3,3
3,"List(Ravi, Kumar, Azure, 678435, null, India)","Map(product -> volvo, color -> , type -> )",6,3
4,"List(Rajesh, )","Map(product -> toyota, color -> white)",2,2
5,List(Rajesh),Map(product -> mercedes),1,1
6,"List(, , )","Map(product -> hyundai, color -> white, type -> suv, Age -> null)",3,4
7,List(),Map(),0,0
8,null,null,null,null


root
 |-- id: long (nullable = true)
 |-- address: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- properties: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- address_count: integer (nullable = true)
 |-- properties_count: integer (nullable = true)



**2) filter condition**

In [0]:
df_size.filter(col("address_count") > 1).display()

id,address,properties,address_count,properties_count
1,"List(Harish, 561321, Bangalore, India)","Map(product -> bmw, color -> brown, type -> sedan)",4,3
2,"List(Kishore, 557890, )","Map(product -> audi, color -> null, type -> truck)",3,3
3,"List(Ravi, Kumar, Azure, 678435, null, India)","Map(product -> volvo, color -> , type -> )",6,3
4,"List(Rajesh, )","Map(product -> toyota, color -> white)",2,2
6,"List(, , )","Map(product -> hyundai, color -> white, type -> suv, Age -> null)",3,4


**3) conditional logic**

In [0]:
from pyspark.sql.functions import when

In [0]:
df_cond = df \
    .withColumn("address_count", array_size("address")) \
    .withColumn(
    "category",
    when(array_size("address") == 0, "Array is Empty")
    .when(F.col("address").isNull(), "Nulls in Array")
    .when(array_size("address") == 1, "Single item")
    .when((array_size("address") > 1) & (array_size("address") < 5), "Multiple items")
    .otherwise("Unknown")
)
display(df_cond)

id,address,properties,address_count,category
1,"List(Harish, 561321, Bangalore, India)","Map(product -> bmw, color -> brown, type -> sedan)",4,Multiple items
2,"List(Kishore, 557890, )","Map(product -> audi, color -> null, type -> truck)",3,Multiple items
3,"List(Ravi, Kumar, Azure, 678435, null, India)","Map(product -> volvo, color -> , type -> )",6,Unknown
4,"List(Rajesh, )","Map(product -> toyota, color -> white)",2,Multiple items
5,List(Rajesh),Map(product -> mercedes),1,Single item
6,"List(, , )","Map(product -> hyundai, color -> white, type -> suv, Age -> null)",3,Multiple items
7,List(),Map(),0,Array is Empty
8,null,null,null,Nulls in Array


**4) nested arrays**

In [0]:
data = [
    (1, [[1,2], [3,4], [5,6], [7,8], [9,0]]),
    (2, [[1,2], [3,4], [5,6], None]),
    (3, [[1,2], [3,4], [5,6]]),
    (4, [[1,2], [3,4]]),
    (5, [[1,2], []]),
    (6, [[], [], []]),
    (7, [[], None]),
    (8, [[]]),
    (8, [None])
]

df_nest = spark.createDataFrame(data, ["id", "numbers"])

df_nested = df_nest.withColumn("array_count", array_size("numbers"))
display(df_nested)

id,numbers,array_count
1,"List(List(1, 2), List(3, 4), List(5, 6), List(7, 8), List(9, 0))",5
2,"List(List(1, 2), List(3, 4), List(5, 6), null)",4
3,"List(List(1, 2), List(3, 4), List(5, 6))",3
4,"List(List(1, 2), List(3, 4))",2
5,"List(List(1, 2), List())",2
6,"List(List(), List(), List())",3
7,"List(List(), null)",2
8,List(List()),1
8,List(null),1


##### 5) Difference between `size() & array_size()`

In [0]:
data = [("Amar", ["Java","Scala","C++","Python","GCC"], ["Spark","Java","Azure","PySpark","Databricks"], [8], "Bangalore", "Chennai", 25, 7),
        ("Ramesh", ["Python","PySpark","C"], ["ADF"], [11, None, 6], "Hyderabad", "Kochin", 35, 8),
        ("Rani", ["Devops","VB"], ["ApacheSpark","Python"], [5, 6], "Amaravathi", "Noida", 30, 10),
        ("Rakesh", ["SQL","Azure",""], ["","Oracle","Confluence"], [12, 6, 8, 15], "Noida", "Mumbai", 33, 5),
        ("Joshi", ["GCC","Git",""], ["SQL",None,"Git"], None, "Delhi", "Kolkata", 28, 6),
        ("Hari", ["Devops","VB",None],None, [5, 6, 8, 10], "Amaravathi", "Noida", 30, 10),
        ("Ritesh", ["SQL","",None], ["PySpark","",""], [12, 6, 8], "luknow", "Mumbai", 33, 5),
        ("Santhosh", ["",""], [], [2, 6, 5, 8, 9, 2], "Delhi", "Noida", 36, 3),
        ("Smitha", [], ["SQL","Git",""], [8, 0, 9], "Delhi", "Noida", 31, 5),
        ("Karan", None, ["SQL",None], [], "Delhi", "Noida", 21, 9)
        ]

schema = StructType([ 
    StructField("EmpName", StringType(), True), 
    StructField("KnownLanguages", ArrayType(StringType()), True), 
    StructField("NewLanguages", ArrayType(StringType()), True),
    StructField("Rating", ArrayType(IntegerType()), True), 
    StructField("CurrentLocation", StringType(), True), 
    StructField("PreviousLocation", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("Experience", IntegerType(), True)
  ])

df_sample = spark.createDataFrame(data=data, schema=schema)
df_sample.printSchema()
display(df_sample)

root
 |-- EmpName: string (nullable = true)
 |-- KnownLanguages: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- NewLanguages: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- Rating: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- CurrentLocation: string (nullable = true)
 |-- PreviousLocation: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Experience: integer (nullable = true)



EmpName,KnownLanguages,NewLanguages,Rating,CurrentLocation,PreviousLocation,Age,Experience
Amar,"List(Java, Scala, C++, Python, GCC)","List(Spark, Java, Azure, PySpark, Databricks)",List(8),Bangalore,Chennai,25,7
Ramesh,"List(Python, PySpark, C)",List(ADF),"List(11, null, 6)",Hyderabad,Kochin,35,8
Rani,"List(Devops, VB)","List(ApacheSpark, Python)","List(5, 6)",Amaravathi,Noida,30,10
Rakesh,"List(SQL, Azure, )","List(, Oracle, Confluence)","List(12, 6, 8, 15)",Noida,Mumbai,33,5
Joshi,"List(GCC, Git, )","List(SQL, null, Git)",null,Delhi,Kolkata,28,6
Hari,"List(Devops, VB, null)",null,"List(5, 6, 8, 10)",Amaravathi,Noida,30,10
Ritesh,"List(SQL, , null)","List(PySpark, , )","List(12, 6, 8)",luknow,Mumbai,33,5
Santhosh,"List(, )",List(),"List(2, 6, 5, 8, 9, 2)",Delhi,Noida,36,3
Smitha,List(),"List(SQL, Git, )","List(8, 0, 9)",Delhi,Noida,31,5
Karan,null,"List(SQL, null)",List(),Delhi,Noida,21,9


     # 12.2 LTS (includes Apache Spark 3.3.2, Scala 2.12)
     from pyspark.sql.functions import size

     # 15.4 LTS (includes Apache Spark 3.5.0, Scala 2.12)
     from pyspark.sql.functions import array_size

**a) size()**
- The below example **combines** the data from **PresentState and PreviousState** and creates a **new column states**.

In [0]:
df_size = df_sample \
    .select(col("EmpName"), array(col("CurrentLocation"), col("PreviousLocation")).alias("JobLocation")) \
    .withColumn("Size", F.size("JobLocation"))
                   
display(df_size)

EmpName,JobLocation,Size
Amar,"List(Bangalore, Chennai)",2
Ramesh,"List(Hyderabad, Kochin)",2
Rani,"List(Amaravathi, Noida)",2
Rakesh,"List(Noida, Mumbai)",2
Joshi,"List(Delhi, Kolkata)",2
Hari,"List(Amaravathi, Noida)",2
Ritesh,"List(luknow, Mumbai)",2
Santhosh,"List(Delhi, Noida)",2
Smitha,"List(Delhi, Noida)",2
Karan,"List(Delhi, Noida)",2


In [0]:
df_size_mlt = df_sample \
    .withColumn("Arr_Size_01", array("CurrentLocation", "PreviousLocation", size("KnownLanguages").cast("string"))) \
    .withColumn("Arr_Size_02", array(size("Rating"), size("KnownLanguages"), size("NewLanguages")))\
    .withColumn("Arr_Size_03", array(size("KnownLanguages"))) \
    .withColumn("Arr_Size_04", array(size("NewLanguages"))) \
    .select("CurrentLocation", "PreviousLocation", "Rating", "KnownLanguages", "NewLanguages", "Arr_Size_01", "Arr_Size_02", "Arr_Size_03", "Arr_Size_04")

display(df_size_mlt)

CurrentLocation,PreviousLocation,Rating,KnownLanguages,NewLanguages,Arr_Size_01,Arr_Size_02,Arr_Size_03,Arr_Size_04
Bangalore,Chennai,List(8),"List(Java, Scala, C++, Python, GCC)","List(Spark, Java, Azure, PySpark, Databricks)","List(Bangalore, Chennai, 5)","List(1, 5, 5)",List(5),List(5)
Hyderabad,Kochin,"List(11, null, 6)","List(Python, PySpark, C)",List(ADF),"List(Hyderabad, Kochin, 3)","List(3, 3, 1)",List(3),List(1)
Amaravathi,Noida,"List(5, 6)","List(Devops, VB)","List(ApacheSpark, Python)","List(Amaravathi, Noida, 2)","List(2, 2, 2)",List(2),List(2)
Noida,Mumbai,"List(12, 6, 8, 15)","List(SQL, Azure, )","List(, Oracle, Confluence)","List(Noida, Mumbai, 3)","List(4, 3, 3)",List(3),List(3)
Delhi,Kolkata,null,"List(GCC, Git, )","List(SQL, null, Git)","List(Delhi, Kolkata, 3)","List(null, 3, 3)",List(3),List(3)
Amaravathi,Noida,"List(5, 6, 8, 10)","List(Devops, VB, null)",null,"List(Amaravathi, Noida, 3)","List(4, 3, null)",List(3),List(null)
luknow,Mumbai,"List(12, 6, 8)","List(SQL, , null)","List(PySpark, , )","List(luknow, Mumbai, 3)","List(3, 3, 3)",List(3),List(3)
Delhi,Noida,"List(2, 6, 5, 8, 9, 2)","List(, )",List(),"List(Delhi, Noida, 2)","List(6, 2, 0)",List(2),List(0)
Delhi,Noida,"List(8, 0, 9)",List(),"List(SQL, Git, )","List(Delhi, Noida, 0)","List(3, 0, 3)",List(0),List(3)
Delhi,Noida,List(),null,"List(SQL, null)","List(Delhi, Noida, null)","List(0, null, 2)",List(null),List(2)


**b) array_size**

In [0]:
df_arrySize = df_sample \
    .withColumn("Arr_Size_01", array("CurrentLocation", "PreviousLocation", array_size("KnownLanguages").cast("string"))) \
    .withColumn("Arr_Size_02", array(array_size("Rating"), size("KnownLanguages"), array_size("NewLanguages")))\
    .withColumn("Arr_Size_03", array(array_size("Rating"))) \
    .withColumn("Arr_Size_04", array(array_size("KnownLanguages"))) \
    .withColumn("Arr_Size_05", array(array_size("NewLanguages"))) \
    .withColumn("Arr_Size_06", array(array_size("Rating"), lit(5), lit(8))) \
    .withColumn("Arr_Size_07", array(array_size("KnownLanguages"), lit(6), lit(9))) \
    .withColumn("Arr_Size_08", array(array_size("NewLanguages"), lit(1), lit(7))) \
    .select("KnownLanguages", "NewLanguages", "Rating", "Arr_Size_01", "Arr_Size_02", "Arr_Size_03", "Arr_Size_04", "Arr_Size_05", "Arr_Size_06", "Arr_Size_07", "Arr_Size_08")

display(df_arrySize)

KnownLanguages,NewLanguages,Rating,Arr_Size_01,Arr_Size_02,Arr_Size_03,Arr_Size_04,Arr_Size_05,Arr_Size_06,Arr_Size_07,Arr_Size_08
"List(Java, Scala, C++, Python, GCC)","List(Spark, Java, Azure, PySpark, Databricks)",List(8),"List(Bangalore, Chennai, 5)","List(1, 5, 5)",List(1),List(5),List(5),"List(1, 5, 8)","List(5, 6, 9)","List(5, 1, 7)"
"List(Python, PySpark, C)",List(ADF),"List(11, null, 6)","List(Hyderabad, Kochin, 3)","List(3, 3, 1)",List(3),List(3),List(1),"List(3, 5, 8)","List(3, 6, 9)","List(1, 1, 7)"
"List(Devops, VB)","List(ApacheSpark, Python)","List(5, 6)","List(Amaravathi, Noida, 2)","List(2, 2, 2)",List(2),List(2),List(2),"List(2, 5, 8)","List(2, 6, 9)","List(2, 1, 7)"
"List(SQL, Azure, )","List(, Oracle, Confluence)","List(12, 6, 8, 15)","List(Noida, Mumbai, 3)","List(4, 3, 3)",List(4),List(3),List(3),"List(4, 5, 8)","List(3, 6, 9)","List(3, 1, 7)"
"List(GCC, Git, )","List(SQL, null, Git)",null,"List(Delhi, Kolkata, 3)","List(null, 3, 3)",List(null),List(3),List(3),"List(null, 5, 8)","List(3, 6, 9)","List(3, 1, 7)"
"List(Devops, VB, null)",null,"List(5, 6, 8, 10)","List(Amaravathi, Noida, 3)","List(4, 3, null)",List(4),List(3),List(null),"List(4, 5, 8)","List(3, 6, 9)","List(null, 1, 7)"
"List(SQL, , null)","List(PySpark, , )","List(12, 6, 8)","List(luknow, Mumbai, 3)","List(3, 3, 3)",List(3),List(3),List(3),"List(3, 5, 8)","List(3, 6, 9)","List(3, 1, 7)"
"List(, )",List(),"List(2, 6, 5, 8, 9, 2)","List(Delhi, Noida, 2)","List(6, 2, 0)",List(6),List(2),List(0),"List(6, 5, 8)","List(2, 6, 9)","List(0, 1, 7)"
List(),"List(SQL, Git, )","List(8, 0, 9)","List(Delhi, Noida, 0)","List(3, 0, 3)",List(3),List(0),List(3),"List(3, 5, 8)","List(0, 6, 9)","List(3, 1, 7)"
null,"List(SQL, null)",List(),"List(Delhi, Noida, null)","List(0, null, 2)",List(0),List(null),List(2),"List(0, 5, 8)","List(null, 6, 9)","List(2, 1, 7)"


In [0]:
# filter: Rows where array has more than 1 element
# Fix: Compare the first element of the array to 1
# Assumes Arr_Size_04 is always a single-element array

df_arrySize.filter(F.col("Arr_Size_04")[0] > 1).display()

KnownLanguages,NewLanguages,Rating,Arr_Size_01,Arr_Size_02,Arr_Size_03,Arr_Size_04,Arr_Size_05,Arr_Size_06,Arr_Size_07,Arr_Size_08
"List(Java, Scala, C++, Python, GCC)","List(Spark, Java, Azure, PySpark, Databricks)",List(8),"List(Bangalore, Chennai, 5)","List(1, 5, 5)",List(1),List(5),List(5),"List(1, 5, 8)","List(5, 6, 9)","List(5, 1, 7)"
"List(Python, PySpark, C)",List(ADF),"List(11, null, 6)","List(Hyderabad, Kochin, 3)","List(3, 3, 1)",List(3),List(3),List(1),"List(3, 5, 8)","List(3, 6, 9)","List(1, 1, 7)"
"List(Devops, VB)","List(ApacheSpark, Python)","List(5, 6)","List(Amaravathi, Noida, 2)","List(2, 2, 2)",List(2),List(2),List(2),"List(2, 5, 8)","List(2, 6, 9)","List(2, 1, 7)"
"List(SQL, Azure, )","List(, Oracle, Confluence)","List(12, 6, 8, 15)","List(Noida, Mumbai, 3)","List(4, 3, 3)",List(4),List(3),List(3),"List(4, 5, 8)","List(3, 6, 9)","List(3, 1, 7)"
"List(GCC, Git, )","List(SQL, null, Git)",null,"List(Delhi, Kolkata, 3)","List(null, 3, 3)",List(null),List(3),List(3),"List(null, 5, 8)","List(3, 6, 9)","List(3, 1, 7)"
"List(Devops, VB, null)",null,"List(5, 6, 8, 10)","List(Amaravathi, Noida, 3)","List(4, 3, null)",List(4),List(3),List(null),"List(4, 5, 8)","List(3, 6, 9)","List(null, 1, 7)"
"List(SQL, , null)","List(PySpark, , )","List(12, 6, 8)","List(luknow, Mumbai, 3)","List(3, 3, 3)",List(3),List(3),List(3),"List(3, 5, 8)","List(3, 6, 9)","List(3, 1, 7)"
"List(, )",List(),"List(2, 6, 5, 8, 9, 2)","List(Delhi, Noida, 2)","List(6, 2, 0)",List(6),List(2),List(0),"List(6, 5, 8)","List(2, 6, 9)","List(0, 1, 7)"
